Q10 — Model part only (Multiple time-series, window=10, 2 inputs/step ⇒ outputs: 7 per step)

In [ ]:
# Q10: model-only (TF/Keras). Input windows: (timesteps=10, features=2) → output 7 features each step.
import tensorflow as tf
from tensorflow.keras import layers, models

inputs = layers.Input(shape=(10, 2))
x = layers.SimpleRNN(17, return_sequences=True)(inputs)   # 17 hidden states (can change)
outputs = layers.TimeDistributed(layers.Dense(7))(x)      # 7 outputs per time step
model = models.Model(inputs, outputs)
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 2)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 10, 17)         │           340 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 10, 7)          │           126 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

Total params: 466 (1.82 KB)

Trainable params: 466 (1.82 KB)

Non-trainable params: 0 (0.00 B)

Q12 — List indexing output

In [ ]:
# Q12
import string
f = list(string.ascii_lowercase)           # ['a', 'b', ..., 'z']
r = f[::-1]                                # reverse (not used here)
print(f[f.index('x') - 20])                # expected: 'd'


d


Q13 — Reshape and index

In [ ]:
# Q13
import numpy as np, string
x = [0]*(1000-26) + list(string.ascii_lowercase)   # last 26 are 'a'..'z'
y = np.reshape(np.array(x, dtype=object), (100, 10))
print(y[99, 6])                                    # index 996 ⇒ within last-26 ⇒ 'w'


w


Q14 — Two-class CNN (your digit=7 ⇒ classes: Baby vs Dog)

In [ ]:
# A) Setup
import tensorflow as tf, matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DATA_DIR = Path('/content/drive/MyDrive/baby_vs_dog')  # <-- keep your folder here
IMG_SIZE = (50, 50)
BATCH = 64
SEED = 123


Mounted at /content/drive


In [ ]:
# B) Load data
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)


Found 415 files belonging to 2 classes.
Using 332 files for training.
Found 415 files belonging to 2 classes.
Using 83 files for validation.
Classes: ['baby', 'dog']


In [ ]:
# C) Plot a few images
plt.figure(figsize=(6,4))
for images, labels in train_ds.take(1):
    for i in range(6):
        ax = plt.subplot(2, 3, i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.show()


<image output omitted for repo size>

In [ ]:
# D) Shared pieces
norm = layers.Rescaling(1./255)
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

early = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy')
rlrop = tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, verbose=1, monitor='val_loss')


In [ ]:
# E) Model A: with subsampling
model_sub = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    norm, augment,
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Conv2D(128, 3, activation='relu'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model_sub.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
hist_sub = model_sub.fit(train_ds, epochs=25, validation_data=val_ds, callbacks=[early, rlrop], verbose=1)
print("Subsampling model val acc:", model_sub.evaluate(val_ds, verbose=0)[1])


Epoch 1/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - accuracy: 0.5012 - loss: 0.6833 - val_accuracy: 0.4819 - val_loss: 0.6838 - learning_rate: 0.0010
Epoch 2/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5544 - loss: 0.6746 - val_accuracy: 0.5904 - val_loss: 0.6534 - learning_rate: 0.0010
Epoch 3/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5961 - loss: 0.6520 - val_accuracy: 0.8193 - val_loss: 0.5902 - learning_rate: 0.0010
Epoch 4/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6831 - loss: 0.5954 - val_accuracy: 0.6386 - val_loss: 0.5877 - learning_rate: 0.0010
Epoch 5/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8096 - loss: 0.5071 - val_accuracy: 0.7590 - val_loss: 0.4982 - learning_rate: 0.0010
Epoch 6/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8073 - loss: 0.4422 - val_accuracy: 0.7349 - val_loss: 0.5322 - learning_rate: 0.0010
Epoch 7/25
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8333 - loss: 0.3924
Epoch 7: ReduceLROnPlate

In [ ]:
# F) Save
model_sub.save('/content/Q14_islam_subsampling.h5')


In [ ]:
# G) Model B: without subsampling
model_nopool = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    norm, augment,
    layers.Conv2D(32, 3, activation='relu'),
    layers.Conv2D(64, 3, activation='relu'),
    layers.Conv2D(128, 3, activation='relu'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model_nopool.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
hist_nopool = model_nopool.fit(train_ds, epochs=25, validation_data=val_ds, callbacks=[early, rlrop], verbose=1)
print("No-pool model val acc:", model_nopool.evaluate(val_ds, verbose=0)[1])


Epoch 1/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 168ms/step - accuracy: 0.5126 - loss: 0.6852 - val_accuracy: 0.5181 - val_loss: 0.6785 - learning_rate: 0.0010
Epoch 2/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.5681 - loss: 0.6805 - val_accuracy: 0.5663 - val_loss: 0.6686 - learning_rate: 0.0010
Epoch 3/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5949 - loss: 0.6605 - val_accuracy: 0.6265 - val_loss: 0.6431 - learning_rate: 0.0010
Epoch 4/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6374 - loss: 0.6252 - val_accuracy: 0.7108 - val_loss: 0.5914 - learning_rate: 0.0010
Epoch 5/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8082 - loss: 0.5334 - val_accuracy: 0.7711 - val_loss: 0.5281 - learning_rate: 0.0010
Epoch 6/25
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.7531 - loss: 0.4745 - val_accuracy: 0.6867 - val_loss: 0.5390 - learning_rate: 0.0010
Epoch 7/25
4/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7410 - loss: 0.5572
Epoch 7: ReduceLROnPla

In [ ]:
# H) Save
model_nopool.save('/content/Q14_islam_nopooling.h5')


Q15 — Five auto stocks, SimpleRNN then LSTM (predict Toyota 'TM')

In [3]:
# === Q15 (Fixed): Predict Toyota (TM) next-day price using 5 auto stocks
# Handles yfinance's 'Adj Close' / 'Close' change. Runs end-to-end with SimpleRNN & LSTM.

!pip -q install yfinance

import yfinance as yf
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models

# ---------------------------------------------------------------------
# 1) Download five auto stocks (daily). Be explicit about auto_adjust.
# ---------------------------------------------------------------------
TICKERS = ['TM','F','GM','HMC','TSLA']    # target = TM
raw = yf.download(TICKERS, period='3y', interval='1d', auto_adjust=False, progress=False)

# Pick price table robustly (Adj Close preferred; fallback to Close)
if isinstance(raw.columns, pd.MultiIndex):
    if 'Adj Close' in raw.columns.get_level_values(0):
        price = raw['Adj Close']
    else:
        price = raw['Close']
else:
    # Single-level columns (rare for multi-ticker); use Close
    price = raw[['Adj Close']] if 'Adj Close' in raw.columns else raw[['Close']]

price = price.dropna().astype('float32')
print("Using columns:", list(price.columns))
print("Shape:", price.shape)
display(price.tail())

# ---------------------------------------------------------------------
# 2) Scale all features to [0,1]; also fit a scaler only for TM to invert later
# ---------------------------------------------------------------------
tgt = 'TM'
scaler_all = MinMaxScaler()
scaled = pd.DataFrame(scaler_all.fit_transform(price), index=price.index, columns=price.columns)

scaler_tgt = MinMaxScaler()
scaler_tgt.fit(price[[tgt]])   # fit only on Toyota column for clean inverse later

# ---------------------------------------------------------------------
# 3) Create windows: X=(WINDOW days, all 5 series), y=next-day TM
# ---------------------------------------------------------------------
WINDOW, HORIZON = 20, 1
vals = scaled.values
X, y = [], []
tgt_idx = scaled.columns.get_loc(tgt)

for i in range(len(vals) - WINDOW - HORIZON + 1):
    X.append(vals[i:i+WINDOW, :])                       # (20, 5)
    y.append(vals[i+WINDOW:i+WINDOW+HORIZON, tgt_idx])  # (1,)

X = np.array(X, dtype=np.float32)                       # (N, 20, 5)
y = np.array(y, dtype=np.float32).reshape(-1, HORIZON)  # (N, 1)
print("X, y shapes:", X.shape, y.shape)

# ---------------------------------------------------------------------
# 4) Chronological split
# ---------------------------------------------------------------------
split = int(0.8 * len(X))
Xtr, Xte = X[:split], X[split:]
ytr, yte = y[:split], y[split:]
dates_te = price.index[WINDOW + split : WINDOW + split + len(yte)]

# ---------------------------------------------------------------------
# 5) Define models
# ---------------------------------------------------------------------
def build_simple_rnn(units=57):
    inp = layers.Input(shape=(WINDOW, X.shape[-1]))
    x = layers.SimpleRNN(units, activation='relu')(inp)
    out = layers.Dense(1, activation='linear')(x)  # predict TM only
    m = models.Model(inp, out)
    m.compile(optimizer='adam', loss='mse')
    return m

def build_lstm(units=57):
    inp = layers.Input(shape=(WINDOW, X.shape[-1]))
    x = layers.LSTM(units)(inp)
    out = layers.Dense(1)(x)
    m = models.Model(inp, out)
    m.compile(optimizer='adam', loss='mse')
    return m

# ---------------------------------------------------------------------
# 6) Train SimpleRNN
# ---------------------------------------------------------------------
rnn = build_simple_rnn()
rnn.fit(Xtr, ytr, epochs=20, batch_size=64, validation_split=0.1, verbose=0)
pred_rnn = rnn.predict(Xte, verbose=0).ravel()

# ---------------------------------------------------------------------
# 7) Train LSTM
# ---------------------------------------------------------------------
lstm = build_lstm()
lstm.fit(Xtr, ytr, epochs=20, batch_size=64, validation_split=0.1, verbose=0)
pred_lstm = lstm.predict(Xte, verbose=0).ravel()

# ---------------------------------------------------------------------
# 8) Inverse-scale predictions/targets back to $ for plotting
# ---------------------------------------------------------------------
true_tm      = scaler_tgt.inverse_transform(yte.reshape(-1,1)).ravel()
pred_rnn_tm  = scaler_tgt.inverse_transform(pred_rnn.reshape(-1,1)).ravel()
pred_lstm_tm = scaler_tgt.inverse_transform(pred_lstm.reshape(-1,1)).ravel()

# ---------------------------------------------------------------------
# 9) Plots
# ---------------------------------------------------------------------
plt.figure(figsize=(10,4))
plt.plot(dates_te, true_tm, label='True TM', linewidth=2)
plt.plot(dates_te, pred_rnn_tm, label='SimpleRNN Pred', alpha=0.9)
plt.title('Toyota (TM) — SimpleRNN (inputs: TM, F, GM, HMC, TSLA)')
plt.xlabel('Date'); plt.ylabel('Adj Close ($)'); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(10,4))
plt.plot(dates_te, true_tm, label='True TM', linewidth=2)
plt.plot(dates_te, pred_lstm_tm, label='LSTM Pred', alpha=0.9)
plt.title('Toyota (TM) — LSTM (inputs: TM, F, GM, HMC, TSLA)')
plt.xlabel('Date'); plt.ylabel('Adj Close ($)'); plt.legend(); plt.tight_layout(); plt.show()

# (Optional) Scaled-space MSE for quick comparison
mse_rnn  = np.mean((yte.ravel() - pred_rnn)**2)
mse_lstm = np.mean((yte.ravel() - pred_lstm)**2)
print(f"Scaled-space MSE — SimpleRNN: {mse_rnn:.6f} | LSTM: {mse_lstm:.6f}")


Using columns: ['F', 'GM', 'HMC', 'TM', 'TSLA']
Shape: (753, 5)


Ticker,F,GM,HMC,TM,TSLA
Date,,,,,
2025-10-17,11.92,58.380001,30.820000,197.830002,439.309998
2025-10-20,11.99,58.000000,31.280001,200.110001,447.429993
2025-10-21,12.56,66.620003,31.280001,201.949997,442.600006
2025-10-22,12.43,67.309998,31.840000,203.509995,438.970001
2025-10-23,12.34,66.849998,31.590000,204.059998,448.980011


X, y shapes: (733, 20, 5) (733, 1)


<image output omitted for repo size>

<image output omitted for repo size>

Scaled-space MSE — SimpleRNN: 0.002410 | LSTM: 0.003552
